In [1]:
# Our 7 target classes with consistent naming
CLASSES = {
    'Vitiligo': 0,
    'Melasma': 1,
    'Psoriasis': 2,
    'Eczema': 3,
    'Tinea': 4,
    'Contact Dermatitis': 5,
    'Seborrheic Dermatitis': 6
}

print(CLASSES)

{'Vitiligo': 0, 'Melasma': 1, 'Psoriasis': 2, 'Eczema': 3, 'Tinea': 4, 'Contact Dermatitis': 5, 'Seborrheic Dermatitis': 6}


What it does: Maps every DermaCon-IN disease label to one of our 7 classes, filters out everything irrelevant, and builds image paths
Why: Each dataset has its own naming conventions. This standardizes everything into our unified label system before combining

In [2]:
import pandas as pd
import os

# ── DermaCon-IN ──────────────────────────────────────────────────────
derma = pd.read_csv('../data/raw/DermaCon-IN/METADATA/Skin_Metadata-1.csv')

# Define which Disease_labels map to which of our 7 classes
derma_mapping = {
    'Vitiligo':               ['Vitiligo'],
    'Melasma':                ['Melasma'],
    'Psoriasis':              ['Psoriasis', 'Chronic plaque psoriasis', 'Guttate Psoriasis',
                               'Psoriasis Vulgaris', 'Palmar psoriasis', 
                               'Inverse psoriasis', 'Pustular psoriasis'],
    'Eczema':                 ['Eczema', 'Infected Eczema', 'Ear Eczema',
                               'Disseminated Eczema', 'Dry Discoid Eczema',
                               'Crusted eczematous dermatitis', 
                               'Chronic eczema with secondary infection'],
    'Tinea':                  ['Tinea Cruris', 'Tinea Corporis', 'Tinea Faciei',
                               'Steroid Modified Tinea', 'Tinea Versicolor',
                               'Tinea Capitis', 'Tinea', 'Tinea Manuum', 
                               'Tinea pedis', 'Infected Tinea'],
    'Contact Dermatitis':     ['Contact Dermatitis', 'Allergic Contact Dermatitis',
                               'Contact Dermatitis with secondary infection'],
    'Seborrheic Dermatitis':  ['Seborrheic Dermatitis'],
}

# Build a reverse lookup: Disease_label → our class name
reverse_mapping = {}
for class_name, labels in derma_mapping.items():
    for label in labels:
        reverse_mapping[label] = class_name

# Filter and remap
derma_filtered = derma[derma['Disease_label'].isin(reverse_mapping.keys())].copy()
derma_filtered['unified_label'] = derma_filtered['Disease_label'].map(reverse_mapping)
derma_filtered['numeric_label'] = derma_filtered['unified_label'].map(CLASSES)
derma_filtered['image_path'] = derma_filtered['Image_name'].apply(
    lambda x: f'../data/raw/DermaCon-IN/Dataset0/{x}' 
    if os.path.exists(f'../data/raw/DermaCon-IN/Dataset0/{x}') 
    else f'../data/raw/DermaCon-IN/Dataset1/{x}'
)

print(f"DermaCon-IN filtered: {len(derma_filtered)} images")
print(derma_filtered['unified_label'].value_counts())

DermaCon-IN filtered: 2688 images
unified_label
Tinea                    1289
Vitiligo                  601
Eczema                    256
Psoriasis                 227
Contact Dermatitis        175
Melasma                    80
Seborrheic Dermatitis      60
Name: count, dtype: int64


What it does: SCIN stores labels as Python lists inside strings so we use ast.literal_eval to parse them properly, then take the top label and map to our 7 classes
Why: SCIN's label format is different from DermaCon-IN where each image has multiple possible labels with confidence scores, so we take the highest confidence one

In [31]:
scin = pd.read_csv('../data/raw/SCIN/scin_labels.csv', dtype={'case_id': str})
scin_cases = pd.read_csv('../data/raw/SCIN/scin_cases.csv', dtype={'case_id': str})

# Merge labels with cases to get actual image paths
scin = scin.merge(scin_cases[['case_id', 'image_1_path']], on='case_id', how='left')

scin['top_label'] = scin['dermatologist_skin_condition_on_label_name'].apply(extract_top_label)

scin_filtered = scin[scin['top_label'].isin(scin_reverse.keys())].copy()
scin_filtered['unified_label'] = scin_filtered['top_label'].map(scin_reverse)
scin_filtered['numeric_label'] = scin_filtered['unified_label'].map(CLASSES)
scin_filtered['image_path'] = scin_filtered['image_1_path'].apply(
    lambda x: f'../data/raw/SCIN/images/{os.path.basename(x)}' if isinstance(x, str) else None
)

print(f"SCIN filtered: {len(scin_filtered)} images")
print(scin_filtered['unified_label'].value_counts())

# Verify paths
for path in scin_filtered['image_path'].head(5):
    print(f"{path} — {'EXISTS' if os.path.exists(path) else 'MISSING'}")

SCIN filtered: 889 images
unified_label
Eczema                   470
Tinea                    154
Psoriasis                126
Contact Dermatitis       124
Seborrheic Dermatitis      7
Melasma                    5
Vitiligo                   3
Name: count, dtype: int64
../data/raw/SCIN/images/-217828380359571871.png — EXISTS
../data/raw/SCIN/images/-3060870142909393201.png — EXISTS
../data/raw/SCIN/images/-1306941150253534667.png — EXISTS
../data/raw/SCIN/images/-3933475004882152757.png — EXISTS
../data/raw/SCIN/images/-1979417173631887595.png — EXISTS


In [24]:
print(scin['case_id'].head(10))

0    -1000600354148496558
1    -1002039107727665188
2    -1003358831658393077
3    -1003826561155964328
4    -1003844406100696311
5    -1005079160214352144
6    -1010778459521153386
7    -1013831220015814987
8     -101827005996397499
9    -1022162013984621110
Name: case_id, dtype: str


In [32]:
# ── SkinDisNet ──────────────────────────────────────────────────────
skindiset = pd.read_csv('../data/raw/SkinDisNet/SkinDisNet/SkinDisNet_part2/SkinDisNet_Metadata.csv')

skindiset_mapping = {
    'Contact Dermatitis':    ['Contact Dermatitis'],
    'Eczema':                ['Eczema'],
    'Seborrheic Dermatitis': ['Seborrheic Dermatitis'],
    'Tinea':                 ['Tinea Corporis'],
}

skindiset_reverse = {}
for class_name, labels in skindiset_mapping.items():
    for label in labels:
        skindiset_reverse[label] = class_name

skindiset_filtered = skindiset[skindiset['Diagnosis'].isin(skindiset_reverse.keys())].copy()
skindiset_filtered['unified_label'] = skindiset_filtered['Diagnosis'].map(skindiset_reverse)
skindiset_filtered['numeric_label'] = skindiset_filtered['unified_label'].map(CLASSES)
skindiset_filtered['image_path'] = skindiset_filtered.apply(
    lambda row: f'../data/raw/SkinDisNet/SkinDisNet/SkinDisNet_part2/Preprocessed/{row["Folder_name"]}/{row["Image_id"]}.jpg',
    axis=1
)

print(f"SkinDisNet filtered: {len(skindiset_filtered)} images")
print(skindiset_filtered['unified_label'].value_counts())

# Verify paths
for path in skindiset_filtered['image_path'].head(5):
    print(f"{path} — {'EXISTS' if os.path.exists(path) else 'MISSING'}")
    

SkinDisNet filtered: 1297 images
unified_label
Contact Dermatitis       477
Eczema                   466
Tinea                    275
Seborrheic Dermatitis     79
Name: count, dtype: int64
../data/raw/SkinDisNet/SkinDisNet/SkinDisNet_part2/Preprocessed/Contact Dermatitis (CD)/CD (1).jpg — EXISTS
../data/raw/SkinDisNet/SkinDisNet/SkinDisNet_part2/Preprocessed/Contact Dermatitis (CD)/CD (2).jpg — EXISTS
../data/raw/SkinDisNet/SkinDisNet/SkinDisNet_part2/Preprocessed/Contact Dermatitis (CD)/CD (3).jpg — EXISTS
../data/raw/SkinDisNet/SkinDisNet/SkinDisNet_part2/Preprocessed/Contact Dermatitis (CD)/CD (4).jpg — EXISTS
../data/raw/SkinDisNet/SkinDisNet/SkinDisNet_part2/Preprocessed/Contact Dermatitis (CD)/CD (5).jpg — EXISTS


In [33]:
# ── Combine all three datasets ────────────────────────────────────────
import pandas as pd

# Keep only the columns we need from each
cols = ['image_path', 'unified_label', 'numeric_label']

derma_final = derma_filtered[cols].copy()
derma_final['source'] = 'DermaCon-IN'

scin_final = scin_filtered[cols].copy()
scin_final['source'] = 'SCIN'

skindiset_final = skindiset_filtered[cols].copy()
skindiset_final['source'] = 'SkinDisNet'

# Combine
master_df = pd.concat([derma_final, scin_final, skindiset_final], ignore_index=True)

print(f"Total images: {len(master_df)}")
print(f"\nPer class:")
print(master_df['unified_label'].value_counts())
print(f"\nPer source:")
print(master_df['source'].value_counts())

Total images: 4874

Per class:
unified_label
Tinea                    1718
Eczema                   1192
Contact Dermatitis        776
Vitiligo                  604
Psoriasis                 353
Seborrheic Dermatitis     146
Melasma                    85
Name: count, dtype: int64

Per source:
source
DermaCon-IN    2688
SkinDisNet     1297
SCIN            889
Name: count, dtype: int64


In [34]:
import os
os.makedirs('../data/processed', exist_ok=True)
master_df.to_csv('../data/processed/master_dataset.csv', index=False)
print("Saved to data/processed/master_dataset.csv")

Saved to data/processed/master_dataset.csv


In [35]:
from sklearn.model_selection import train_test_split

# First split off test set (20%)
train_val_df, test_df = train_test_split(
    master_df, 
    test_size=0.2, 
    random_state=42,
    stratify=master_df['numeric_label']
)

# Then split remaining into train and validation (80/20 of remaining)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.2,
    random_state=42,
    stratify=train_val_df['numeric_label']
)

print(f"Train: {len(train_df)} images")
print(f"Validation: {len(val_df)} images")
print(f"Test: {len(test_df)} images")
print(f"\nTrain class distribution:")
print(train_df['unified_label'].value_counts())

Train: 3119 images
Validation: 780 images
Test: 975 images

Train class distribution:
unified_label
Tinea                    1099
Eczema                    763
Contact Dermatitis        497
Vitiligo                  386
Psoriasis                 226
Seborrheic Dermatitis      94
Melasma                    54
Name: count, dtype: int64


In [36]:
train_df.to_csv('../data/processed/train.csv', index=False)
val_df.to_csv('../data/processed/val.csv', index=False)
test_df.to_csv('../data/processed/test.csv', index=False)

print("Saved:")
print(f"  train.csv — {len(train_df)} images")
print(f"  val.csv   — {len(val_df)} images")
print(f"  test.csv  — {len(test_df)} images")

Saved:
  train.csv — 3119 images
  val.csv   — 780 images
  test.csv  — 975 images
